# Unifying APIs with LiteLLM

Hello everyone! In the real world, the LLM landscape is vast and constantly changing. New, powerful models are released all the time from providers like Google, Anthropic, Meta, and more.

**This presents a major engineering challenge:** how do you use the best model for the job without rewriting your code every time? This is the problem that a brilliant open-source tool called **LiteLLM** solves.

LiteLLM is a **unified translation layer**. It allows you to call over 100 different LLM providers using the exact same format you've already learned. You write your code once, and LiteLLM handles the complex work of translating your request to the specific format each provider needs.

This notebook will demonstrate how to use LiteLLM to seamlessly switch between different AI providers.

## Installation and Setup

First, we need to install the `litellm` library. We also need to remember our security rule: each new provider will require its own API key, which must be set as an environment variable.

> Important: To run the final example in this notebook, you will need an API key from another provider, such as Anthropic (for Claude models). You would add this to your `.env` file just like you did for OpenAI.
> 
> 
> **Example `.env` file:**
> 
> ```
> OPENAI_API_KEY="sk-..."
> ANTHROPIC_API_KEY="sk-ant-..."
> 
> ```

In [38]:
import openai
import litellm
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()

print("Libraries installed and environment variables loaded.")

Libraries installed and environment variables loaded.


In [39]:
OPENAI_MODEL="gpt-4o-mini"
MESSAGES = [
    {
    "role": "user",
    "content": "Tell me a fun fact about Golang."
    }
]

## The Baseline - A Standard OpenAI Call

Let's start with a standard API call using the `openai` library. This is our familiar baseline. We'll ask a simple question to see the standard response.

In [40]:
client = openai.OpenAI()
response_openai = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=MESSAGES
)

In [41]:
display(Markdown(response_openai.choices[0].message.content))

A fun fact about Go (often referred to as Golang) is that it was originally created at Google by Robert Griesemer, Rob Pike, and Ken Thompson in 2007, and it was publicly announced in 2009. One of its primary motivations was to address shortcomings of other languages in terms of performance and ease of use when developing large-scale software, especially in cloud computing. Interestingly, Go's syntax was influenced by several programming languages, including C, but it intentionally avoids inclusion of features like inheritance, which is often found in object-oriented languages. This design choice contributes to Go’s simplicity and efficiency!

## LiteLLM Calling an OpenAI Model

Now, let's make the exact same call, but this time using `litellm`. Notice two things:

1. We call the `litellm.completion()` function.
2. The `model` string is now prefixed with the provider: `"openai/gpt-4o-mini"`.

The rest of the code structure is identical.

In [ ]:
response_litellm = litellm.completion(
    model=f"openai/{OPENAI_MODEL}",
    messages=MESSAGES
)

In [25]:
display(Markdown(response_litellm.choices[0].message.content))

A fun fact about Golang (or Go) is that it was developed at Google in just three years, starting in 2007, and was publicly announced in 2009. One of its main motivations was to address shortcomings in other programming languages used at Google, particularly in terms of compilation speed and ease of use for concurrent programming. The creators, Robert Griesemer, Rob Pike, and Ken Thompson, designed Go to be simple, efficient, and to make building scalable software easier—focusing on networked applications. As a result, it's widely used for cloud services, DevOps tools, and microservices!

## The Real Power - Switching to Another Provider

This is where LiteLLM's magic shines. We are now going to call a completely different model from a different company: Anthropic's Claude 3.5 Haiku.

The **only** thing we change in our code is the `model` string. We don't change the function call, the parameters, or the way we parse the response. As long as our `ANTHROPIC_API_KEY` is set in our `.env` file, LiteLLM handles everything.

In [ ]:
# OpenAI models
valid_openai_models = litellm.get_valid_models(
    custom_llm_provider="openai",
    check_provider_endpoint=True
)
print(f"Valid OpenAI models: {valid_openai_models}")

['anthropic/claude-haiku-5-5', 'anthropic/claude-sonnet-5-5', 'anthropic/claude-opus-5-5', 'anthropic/claude-fable-5-1', 'anthropic/claude-opus-5', 'anthropic/claude-sonnet-5', 'anthropic/claude-fable-5', 'anthropic/claude-opus-4-8', 'anthropic/claude-opus-4-7', 'anthropic/claude-sonnet-4-6', 'anthropic/claude-opus-4-6', 'anthropic/claude-opus-4-5-20251101', 'anthropic/claude-haiku-4-5-20251001']


In [ ]:
# Anthropic models
valid_anthropic_models = litellm.get_valid_models(
    custom_llm_provider="anthropic",
    check_provider_endpoint=True
)
print(f"Valid Anthropic models: {valid_anthropic_models}")

In [50]:
PROVIDER = "openai"
# PROVIDER = "anthropic"
OPENAI_MODEL="gpt-4o-mini"
ANTHROPIC_MODEL = "claude-haiku-5-5"
MODEL = None
if PROVIDER == "openai":
    MODEL = OPENAI_MODEL
elif PROVIDER == "anthropic":
    MODEL = ANTHROPIC_MODEL

print(f"{PROVIDER}/{MODEL} model will be used for completition.")

openai/gpt-4o-mini model will be used for completition.


In [ ]:
response_litellm = litellm.completion(
    model=f"{PROVIDER}/{MODEL}",
    messages=MESSAGES
)

In [52]:
print(f"Response from {PROVIDER}/{MODEL}")
display(Markdown(response_litellm.choices[0].message.content))

Response from openai/gpt-4o-mini


One fun fact about Golang (or Go) is that it was developed at Google by Robert Griesemer, Rob Pike, and Ken Thompson, and it was officially unveiled to the public in 2009. A quirky aspect of its creation is that it was designed to improve programming productivity in an era when software engineering was becoming increasingly complex. Despite its simplicity and ease of use, Go has features like garbage collection and built-in support for concurrent programming using goroutines, making it a powerful language for modern software development. The language's mascot, an adorable gopher, has gained a significant following within the community, leading to a variety of fan art and merchandise!